# RUN_MCDA - multi-criteria ranking of pLCA scenarios

The driver for the `mcda/` package: every formula lives in `mcda/`, every setting
in `mcda/config.py`. What is here is the sequence of calls and the tables worth
looking at on the way.

**What this needs.** One input file, `scenario_summary.xlsx`, and `numpy` /
`pandas` / `pulp`. It touches no database and imports no Brightway: each row of
`scenario_summary.xlsx` is one whole completed pLCA run, and the table is
assembled from several of them by hand (`blank_inputs/scenario_summary_blank.xlsx`
is the template).

**What it writes**, both into this folder:

| file | contents |
|---|---|
| `integrated_mcda_results.xlsx` | all 27 frames — every intermediate, for checking the working |
| `integrated_mcda_results_short.xlsx` | the 3 that get reported — the matrix, the weights, the ranking |

**To change the study rather than the data**, edit `mcda/config.py`: the expert
panel and its weights, which criteria are costs, the ±10% fuzzification width
and the three CoCoSo coefficients are all named there.

## 0. The package

In [ ]:
import mcda

## 1. The decision matrix

One row per alternative — each is a completed pLCA scenario — and one column per
criterion. The first column holds the alternative names; the rest are the
criteria, and **their order is what `mcda/config.py` is aligned to**:
`CRITERION_TYPES` says which are costs, and each expert's list of judgements
runs down the criteria in this same order.

**Where the file comes from.** It is assembled by hand from several completed
pLCA runs — one row per run — and is written by nothing in this repository.
`blank_inputs/scenario_summary_blank.xlsx` is the empty template, and
`Examples/MCDA_2030_SSP2-1000/scenario_summary.xlsx` a worked example: copy
either beside this notebook as `scenario_summary.xlsx`.

Point `SCENARIO_SUMMARY` at any path to use a different study.

In [ ]:
SCENARIO_SUMMARY = "scenario_summary.xlsx"

D_df = mcda.integrated.load_decision_matrix(SCENARIO_SUMMARY)
D_df

## 2. The m-ITARA thresholds

None of these three is an independent input — each is a fixed multiple of one of
the decision matrix's own column statistics, so a new dataset needs no new
numbers. The first three columns below are the statistics, the last three the
thresholds built from them:

| | | |
|---|---|---|
| `AL` | aspiration level | `0.50 × min` — a notional ideal, appended to the matrix as an extra alternative |
| `WL` | worst level | `1.25 × max` |
| `IT` | indifference threshold | `0.50 × std` (population) — the gap below which two alternatives count as indistinguishable |

`IT` is the one that changes the answer most: a criterion is only credited for
separations wider than half its own spread.

> All criteria are assumed to be costs (lower is better): `AL` comes from the
> column minimum and `WL` from the maximum.

In [ ]:
thresholds = mcda.integrated.derive_thresholds(D_df)
thresholds.round(4)

## 3. The expert panel

Seven experts, each rating all eleven criteria on the nine-point linguistic
scale. This is the only subjective input to the whole tool, and the only thing
`mcda/opa.py` reads. Bound here to the decision matrix's own criterion names.

In [ ]:
mcda.config.linguistic_frame(D_df.columns)

## 4. Run

Three weighting stages feed one ranking stage — objective weights from the
matrix (m-ITARA I + II), subjective weights from the panel (fuzzy OPA, solved
as a linear programme by CBC), the two fused by EWCS, then fuzzy
CoCoSo-Bonferroni on the matrix widened to ±10%.

Everything else it needs — the criterion types, the panel, the expert weights,
`epsilon`, `phi1`, `phi2` and `pi_coef` — comes from `mcda/config.py`.

In [ ]:
results = mcda.integrated.analyse(D_df, thresholds)

## 5. The answer

### Final composite criterion weights

In [ ]:
results["Final_Weights"].round(6)

### Final alternative ranking

`Crisp` and `K` hold the same number; both are kept for compatibility.

In [ ]:
results["Ranking"][["Alternative", "Crisp", "Rank"]].round(6)

## 6. Export

Two workbooks, written beside this notebook. Both are overwritten without
asking, so rename them before a second study if the first is worth keeping.

In [ ]:
mcda.integrated.export_results_to_excel(results, "integrated_mcda_results.xlsx")

In [ ]:
mcda.integrated.export_results_to_excel(mcda.integrated.short_results(results),
                                        "integrated_mcda_results_short.xlsx")

---

### Notes

- `AL` / `WL` / `IT` are derived by `mcda.integrated.derive_thresholds()` and
  returned as one table, so the derivation is visible next to the matrix it
  comes from.
- The full workbook keeps every intermediate; the short one holds only what is
  reported.